# Splitting the corpus

One partition, made once, that every later stage reads: CPT, SFT, DPO and the
benchmarks. A paper trained on in one stage can then never be evaluated on in
another.

| split | what it is for |
|---|---|
| **train** | raw papers into CPT; derived SFT and DPO examples into training |
| **evaluate** | absent from all training; used while iterating |
| **test** | locked, untouched until the final evaluation |

The corpus was already deduplicated by study family before download, so each
`family_id` holds one paper and a paper-level split is a family-level split.
The audit cell looks for near-identical titles across splits anyway, in case
anything slipped through.

Run in order. Nothing is written until the last cell.


In [5]:
# ============================ read the corpus ================================
import importlib
import sys
from pathlib import Path

import pandas as pd

DATA = next(
    (folder for start in [Path.cwd(), *Path.cwd().parents]
     for folder in (start, start / "01-data-engineering" / "data-extraction")
     if (folder / "mufasa_corpus" / "parsed" / "markdown").is_dir()),
    None,
)
sys.path.insert(0, str(DATA))
import mufasa_split as splitter
importlib.reload(splitter)

MARKDOWN = DATA / "mufasa_corpus" / "parsed" / "markdown"
RAW = DATA / "extraction_output" / "raw"
OUT_ROOT = DATA / "corpus_splits"

manifest = splitter.read_manifest(MARKDOWN)
extracted = {p.stem for p in RAW.glob("*.json")}
manifest["extracted"] = manifest.paper_id.isin(extracted)

print(f"papers on disk        : {len(manifest):,}")
print(f"distinct family_id    : {manifest.family_id.nunique():,}")
print(f"with extraction JSON  : {int(manifest.extracted.sum()):,}"
      f"   ({100*manifest.extracted.mean():.0f}%)")
print(f"total characters      : {manifest.chars.sum()/1e6:,.0f}M"
      f"   (~{manifest.chars.sum()/4/1e6:,.0f}M tokens)")
print("\nby domain:")
print(manifest.domain.value_counts().to_string())
print("\nby licence:")
print(manifest.licence.value_counts().to_string())


papers on disk        : 10,480
distinct family_id    : 10,480
with extraction JSON  : 10,131   (97%)
total characters      : 423M   (~106M tokens)

by domain:
domain
HLT    3838
ENV    2550
AGR    2401
ENR     787
MAT     599
TEC     305

by licence:
licence
cc-by            10135
cc-by-sa           331
public-domain       14


In [6]:
# ============================== make the split ===============================
# Whole families to one side or the other, drawn separately within each domain
# so a small domain cannot vanish from a split by chance.
EVALUATE = 0.03      # ~315 papers to iterate against
TEST = 0.03          # ~315 papers, locked until the end
FLOOR = 25           # minimum papers per domain per split, so the small domains
                     # (technology, materials) can be scored on their own and
                     # not only inside the pooled number. 0 turns this off.
SEED = 7

manifest = splitter.split(manifest, evaluate=EVALUATE, test=TEST,
                          seed=SEED, floor=FLOOR)

summary = manifest.groupby("split").agg(
    papers=("paper_id", "count"),
    families=("family_id", "nunique"),
    extracted=("extracted", "sum"),
    Mchars=("chars", lambda s: round(s.sum() / 1e6, 1)),
)
summary["Mtokens"] = (summary.Mchars / 4).round(1)
print(summary.to_string())

print()
print("domain balance across splits (% of each split):")
share = pd.crosstab(manifest.split, manifest.domain, normalize="index") * 100
print(share.round(1).to_string())

# The counts matter as much as the shares: a per-domain score needs papers
# behind it. Below ~20 the number moves too much with one paper to trust.
print()
print("papers held out per domain (thin means: report pooled, not per-domain):")
counts = pd.crosstab(manifest.domain, manifest.split)
for domain in counts.index:
    held = counts.loc[domain]
    thin = "   <-- thin" if min(held.get("evaluate", 0), held.get("test", 0)) < 20 else ""
    print(f"   {domain:<5} train {held.get('train', 0):>5,}"
          f"   evaluate {held.get('evaluate', 0):>4}"
          f"   test {held.get('test', 0):>4}{thin}")


          papers  families  extracted  Mchars  Mtokens
split                                                 
evaluate     338       338        325    13.9      3.5
test         338       338        326    13.6      3.4
train       9804      9804       9480   395.1     98.8

domain balance across splits (% of each split):
domain     AGR  ENR   ENV   HLT  MAT  TEC
split                                    
evaluate  21.3  7.4  22.5  34.0  7.4  7.4
test      21.3  7.4  22.5  34.0  7.4  7.4
train     23.0  7.5  24.5  36.8  5.6  2.6

papers held out per domain (thin means: report pooled, not per-domain):
   AGR   train 2,257   evaluate   72   test   72
   ENR   train   737   evaluate   25   test   25
   ENV   train 2,398   evaluate   76   test   76
   HLT   train 3,608   evaluate  115   test  115
   MAT   train   549   evaluate   25   test   25
   TEC   train   255   evaluate   25   test   25


In [7]:
# ================================ audit it ===================================
# Cheap to check, expensive to discover later.
# Heal first: any near-duplicate pair the family dedup missed goes to train,
# so nothing in evaluate or test has a twin the model was trained on.
manifest, moved = splitter.heal(manifest)
if moved:
    print(f"moved {len(moved)} paper(s) to train, to keep their twin out of a held-out split:")
    for m in moved:
        print(f"   {m['paper_id']}  was {m['was']:<8} {m['title']}")
    print()

problems, crossing = splitter.audit(manifest)
if problems:
    print("PROBLEMS")
    for line in problems:
        print("   -", line)
else:
    print("clean: no family spans two splits, every paper assigned, no duplicates")

if crossing:
    print(f"\nnear-identical titles landing in different splits ({len(crossing)}):")
    for key in crossing[:5]:
        rows = manifest[manifest.title.map(lambda t: splitter.title_key(t) == key)]
        for row in rows.itertuples():
            print(f"   [{row.split:<8}] {row.paper_id}  {str(row.title)[:76]}")
        print()
    print("   These would make a held-out score softer than it looks. Move them")
    print("   all to train, or drop the duplicates, before writing the folders.")


moved 2 paper(s) to train, to keep their twin out of a held-out split:
   W2014642624  was test     Lineaments study using aeromagnetic data over parts of southern Bida b
   W4404837589  was evaluate Design and Construction of Smart Toilet using Bamboo Architecture in B

clean: no family spans two splits, every paper assigned, no duplicates


In [8]:
# ===================== write the manifest and the folders ====================
# MODE: "copy" duplicates the files (safe, uses disk). "link" makes hard links
# on the same volume and costs nothing. Originals are never moved or changed.
MODE = "copy"

counts = splitter.materialise(manifest, MARKDOWN, RAW, OUT_ROOT, mode=MODE)
print(f"folders under {OUT_ROOT}:")
print()
for name in ("train", "evaluate", "test"):
    placed = counts.get(name, {})
    print(f"   {name}/markdown  {placed.get('markdown', 0):>6,} files"
          f"      {name}/raw  {placed.get('raw', 0):>6,} files"
          f"   (no raw for {placed.get('no raw', 0):,},"
          f" stale removed {placed.get('stale', 0):,})")

# The manifest lives INSIDE the split folder it describes, so the partition
# travels as one thing - copy corpus_splits/ to Colab and it comes along.
path = splitter.write_manifest(manifest, OUT_ROOT / "manifest.parquet")
print()
print(f"manifest: {path.relative_to(DATA)}  ({len(manifest):,} rows)")

print()
print("Every later stage should read the manifest, not re-split:")
print("   manifest = pd.read_parquet('corpus_splits/manifest.parquet')")
print("   train_papers = set(manifest[manifest.split == 'train'].paper_id)")


folders under c:\CodingWorld\Hackathons\AfricanDeepTechChallenge\MUFASA\01-data-engineering\data-extraction\corpus_splits:

   train/markdown   9,806 files      train/raw   9,482 files   (no raw for 324)
   evaluate/markdown     337 files      evaluate/raw     324 files   (no raw for 13)
   test/markdown     337 files      test/raw     325 files   (no raw for 12)

manifest: corpus_splits\manifest.parquet  (10,480 rows)

Every later stage should read the manifest, not re-split:
   manifest = pd.read_parquet('corpus_splits/manifest.parquet')
   train_papers = set(manifest[manifest.split == 'train'].paper_id)
